# Systematic Momentum Strategy

Reproducible research workflow for the shared implementation in `src/`. The notebook intentionally contains no duplicated backtest functions and no committed result outputs. Run every cell from the repository root to regenerate results with current market data.

In [ ]:
from pathlib import Path

if not Path("src").is_dir():
    raise RuntimeError("Run this notebook from the repository root.")
%pip install -r requirements.txt -q

In [ ]:
import matplotlib.pyplot as plt

from src.analysis import analyze, investable_signal_date
from src.backtest import equity_curve
from src.data import DEFAULT_UNIVERSE, load_prices
from src.metrics import summary_table

START, END = "2015-01-01", "2025-01-01"
LOOKBACK, TC_BPS, QUANTILE = 12, 10.0, 0.2

In [ ]:
prices = load_prices(DEFAULT_UNIVERSE, START, END)
prices.shape, prices.index.min(), prices.index.max()

In [ ]:
result = analyze(
    prices,
    lookback_months=LOOKBACK,
    tc_bps=TC_BPS,
    quantile=QUANTILE,
)
print(f"Evaluation start: {result.evaluation_start.date()}")
print(f"Average one-way turnover: {result.average_turnover:.1%}")
result.summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
equity_curve(result.momentum["ret"]).plot(ax=ax, label="Momentum (net)")
equity_curve(result.gross["ret"]).plot(
    ax=ax, label="Momentum (gross)", linestyle="--", alpha=0.7
)
equity_curve(result.benchmark["ret"]).plot(ax=ax, label="Equal-weight benchmark")
ax.set(title="Cross-sectional momentum vs. benchmark", ylabel="Growth of $1")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()

## Lookback sensitivity

All variants below use the latest first-signal date among the tested lookbacks, ensuring an identical evaluation window.

In [ ]:
lookbacks = (6, 9, 12)
common_signal_date = max(
    investable_signal_date(prices, lookback, QUANTILE) for lookback in lookbacks
)
sensitivity_runs = {
    lookback: analyze(
        prices,
        lookback_months=lookback,
        tc_bps=TC_BPS,
        quantile=QUANTILE,
        start_signal_date=common_signal_date,
    )
    for lookback in lookbacks
}
summary_table(
    {
        f"Momentum {lookback}-1 (net)": run.momentum["ret"]
        for lookback, run in sensitivity_runs.items()
    }
)